## DNN Pricer

### Simulating GBM (as described in Chapter 4 in the report)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import time

np.random.seed(7)
torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32

#Simulating GBM (as described in Chapter 4 in the report)
def simulate_gbm(S0, d, N, K, r, q, sigma, rho, T, n_dates,
                 device = device, dtype = dtype):

    dt = T/n_dates
    
    S = torch.empty((N,n_dates + 1, d), device = device, dtype = dtype)
    S[:,0,:] = S0

    pi = torch.full((d,d), rho, device = device, dtype = dtype)
    pi.fill_diagonal_(1.0)
    L = torch.linalg.cholesky(pi)                   #cholesky is lower triang decomposition where pi = L*L^T

    for i in range(n_dates):
        z = torch.randn((N,d), device = device, dtype = dtype)
        omega = z @ L.T                              #var(z*L^T) = Lvar(z)LT = L*I*L^T = L*L^T = var(omega)
        S[:,i+1,:] =  S[:,i,:]*torch.exp((r-q-0.5*sigma**2)*dt + sigma*torch.sqrt(torch.tensor(dt, device = device, dtype = dtype))*omega)

    return S


def payoff_max_call(S_t, K):
    return torch.relu(S_t.max(dim = -1).values - K) 

### DNN model and the backward recursion algorithm (Algorithm 1 in Chapter 1)

In [ ]:
#build the dnn architecture 

class DeepNeuralNetwork(nn.Module):
    def __init__(self, d_input):
        super().__init__()
        self.dnn = nn.Sequential(
            torch.nn.Linear(d_input, 150 + d_input),
            torch.nn.ReLU(),
            torch.nn.Linear(150 + d_input, 150 + d_input),
            torch.nn.ReLU(),
            torch.nn.Linear(150 + d_input, 1))
    def forward(self, X):
        z = self.dnn(X).reshape(-1)
        return z

#train dnn by minimising the MSE with ADAM

def train_dnn(X_scaled, y, d, epochs, lr, weight_decay, batch_size = 1024):

    dnn_model = DeepNeuralNetwork(d_input = d).to(device = device, dtype = dtype)
    dnn_model.train()

    #ADAM sgd
    optimiser = torch.optim.AdamW(dnn_model.parameters(), lr = lr, weight_decay = weight_decay)

    loss_dnn = nn.MSELoss()
    
    #mini batch sgd (sgd discussed in chapter 3)
    for epoch in range(epochs):
        
        indices = torch.randperm(X_scaled.shape[0], device = device)

        for i in range(0, X_scaled.shape[0], batch_size):
            idx = indices[i:i + batch_size]
            Xbatch = X_scaled[idx]
            ybatch = y[idx].reshape(-1)
            
            optimiser.zero_grad(set_to_none = True)
            train_pred = dnn_model(Xbatch).reshape(-1)
            comp_loss = loss_dnn(train_pred, ybatch)
            comp_loss.backward()
            optimiser.step()

    return dnn_model

#use dnn to estimate the continuation value

@torch.no_grad()                                                                      #https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html
def dnn_pred(dnn_model, X_scaled):
    dnn_model.eval()
    return dnn_model(X_scaled)


#the standard backward recursion algorithm

def dnn_maxcall_pricer(paths, N, d, K , T, n_dates, r,
                       epochs = 360, lr = 1e-3, weight_decay = 1e-8):

    dt = T/n_dates

    v_cont = payoff_max_call(paths[:,n_dates,:], K = K)                   #value of the option at time T

    for i in range(n_dates - 1,0, -1):                                     #backwards recursion
        X_i = paths[:,i,:]
        y_i = v_cont

        #scale for numerical stability

        x_min = X_i.min(dim = 0).values
        x_max = X_i.max(dim = 0).values

        Xi_scaled =  (X_i - x_min) / (x_max - x_min).clamp_min(1e-14)      #avoid / by 0

        dnn_model = train_dnn(X_scaled = Xi_scaled, y = y_i, d = d, epochs = epochs, lr = lr, weight_decay = weight_decay)

        cont_est = dnn_pred(dnn_model,Xi_scaled)                    #estimated continuation value
        c_hat = torch.exp(torch.tensor(-r * dt, device = device, dtype = dtype))*cont_est     #estimated discounted continuation value

        g_i = payoff_max_call(X_i, K = K)                           #payoff at time i
        exer = (g_i > c_hat) & (g_i > 0)                            #exercise if payoff > estimated continuation value and payoff>0

        v_cont = torch.where(exer, g_i, torch.exp(torch.tensor(-r * dt, device = device, dtype = dtype)) * v_cont)
                                                                    #update cashflows

    v0 = np.exp(-r*dt) * v_cont.mean().item()
    
    return v0

### Compute option value + computational time

In [ ]:
S0 = 110.0             #spot price
T = 3.0                #maturity
K = 100.0              #strike price
n_dates = 9            #num of exercise dates
r = 0.05               #risk free interest rate
q = 0.1                #divident yield
sigma = 0.2            #volatility
rho = 0.0              #correlation
N = 10000              #number of simulated paths
d = 50                 #number of underlyings

paths = simulate_gbm(S0 = S0, d = d, N = N, K = K, r = r, q = q, sigma = sigma, rho = rho,
                     T = T, n_dates = n_dates,
                     device = device, dtype = dtype)

start = time.time()

v = dnn_maxcall_pricer(paths = paths, N = N, d = d, K = K, T = T, n_dates = n_dates, r = r,
                       epochs = 360, lr = 1e-3, weight_decay = 1e-8)
stop = time.time()


print( f"d = {d} | S0 = {S0}| {v:.3f} | time: {stop - start:.2f}s")